# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Instalar cliente de Groq y leer API key desde Colab Secrets

!pip install groq --quiet

from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get("GROQ_API_KEY"))
print("Cliente de Groq inicializado correctamente.")

## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [ ]:
# Prompt de clasificación en modo zero-shot

prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.choices[0].message.content)

In [ ]:
# Prompt de clasificación en modo few-shot

prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.choices[0].message.content)
# El formato few-shot suele acotar mejor la salida a una sola palabra de la categoría esperada

### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [ ]:
# Razonamiento paso a paso (chain-of-thought)

problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)
# Variante sugerida en clase: pedir solo el resultado y comparar la precisión.

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [ ]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina

prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)
# Aunque suene segura, el modelo no tiene una fuente para conocer esta información.

## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [ ]:
# Instalar sentence-transformers

!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings

modelo_embeddings = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

In [ ]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante

def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo que compré en oferta?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

In [ ]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG

prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.choices[0].message.content)
# Con el fragmento recuperado, la respuesta queda fundamentada en la política proporcionada.

# **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

## **INSTRUCCIONES:**

**1. Define tu base de conocimiento y genera embeddings:**

* Lee la API key previamente configurada desde **Colab Secrets** e instala/importa las librerías necesarias.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings con `sentence-transformers`.

In [ ]:
# Leer API key, instalar e importar librerías

!pip install groq sentence-transformers --quiet

from google.colab import userdata
from groq import Groq
from sentence_transformers import SentenceTransformer, util

groq_api_key = userdata.get("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("Agrega GROQ_API_KEY en Google Colab Secrets antes de continuar.")

client = Groq(api_key=groq_api_key)
MODELO_GROQ = "openai/gpt-oss-20b"

In [ ]:
# Definir la lista documentos y generar sus embeddings

documentos = [
    "Política de asistencia: cada estudiante puede tener como máximo tres faltas durante el curso. Una cuarta falta implica perder el derecho a la evaluación final.",
    "Política de entregas: las tareas se reciben hasta 48 horas después de la fecha límite con una penalización del 20 %. Después de ese plazo no se aceptan.",
    "Política de reposición: una evaluación solo puede reponerse si el estudiante presenta un justificante válido dentro de los cinco días naturales posteriores al examen.",
    "Política de integridad académica: copiar o compartir respuestas en una evaluación ocasiona la anulación de esa evaluación y una notificación al comité académico."
]

modelo_embeddings = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings_documentos = modelo_embeddings.encode(documentos, convert_to_tensor=True)

print(f"Se vectorizaron {len(documentos)} fragmentos.")
print("Forma de los embeddings:", tuple(embeddings_documentos.shape))

**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [ ]:
# Definir la función buscar_fragmento

def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode(pregunta, convert_to_tensor=True)
    similitudes = util.cos_sim(embedding_pregunta, embeddings_documentos)[0]
    indice_mas_relevante = similitudes.argmax().item()
    return documentos[indice_mas_relevante]

**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [ ]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag

pregunta = "¿Hasta cuándo puedo presentar el justificante para reponer una evaluación?"

response_sin_rag = client.chat.completions.create(
    model=MODELO_GROQ,
    messages=[{"role": "user", "content": pregunta}]
)
respuesta_sin_rag = response_sin_rag.choices[0].message.content

**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [ ]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag

contexto = buscar_fragmento(pregunta)

prompt_con_rag = f"""Responde la pregunta usando únicamente el contexto proporcionado.
No inventes información ni agregues reglas que no aparezcan en el contexto.
Si el contexto no contiene la respuesta, indica claramente que no hay información suficiente.

Contexto recuperado:
{contexto}

Pregunta:
{pregunta}

Respuesta:"""

response_con_rag = client.chat.completions.create(
    model=MODELO_GROQ,
    messages=[{"role": "user", "content": prompt_con_rag}]
)
respuesta_con_rag = response_con_rag.choices[0].message.content

**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [ ]:
# Mostrar ambas respuestas para comparar

print("PREGUNTA")
print(pregunta)
print("\nCONTEXTO RECUPERADO")
print(contexto)
print("\nRESPUESTA SIN RAG")
print(respuesta_sin_rag)
print("\nRESPUESTA CON RAG")
print(respuesta_con_rag)
print("\nCONCLUSIÓN")
print(
    "La respuesta con RAG está mejor fundamentada porque usa la política de reposición "
    "recuperada por similitud coseno; la respuesta sin RAG no recibe esa política y puede "
    "suponer un plazo distinto."
)